# Coffee17 W0-HBP Matched V1

Eksperimen matched:

`R0 -> MobileNetV3-Large -> HBP -> Linear17`

vs

`W0 -> MobileNetV3-Large -> HBP -> Linear17`

Keduanya dilatih ulang pada **5 fold, seed42**, dengan initialization dan training recipe yang sama. Satu-satunya faktor eksperimen adalah preprocessing input.

Kaggle Input: **Coffee Green Bean with 17 Defects Original** saja.

Outer test tidak disentuh.


In [ ]:
NOTEBOOK_BUILD = "W0-HBP-MATCHED-V1"
SCIENTIFIC_CODE_COMMIT = "5a965f5af813968649b0e82ba5dbc39ef5ba477c"
print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-w0-hbp-matched-code"
PROJECT = WORK / "coffee17-w0-hbp-matched-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Resume hanya dari output notebook matched ini.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-w0-hbp-matched-project")
    if p.is_dir()
):
    print("MERGE PRIOR MATCHED OUTPUT:", prior)
    if PROJECT.exists():
        raise RuntimeError("Lebih dari satu prior matched project ditemukan.")
    shutil.copytree(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact scientific code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

CONFIG = REPO / "configs/w0_hbp_matched/W0_HBP_MATCHED.yaml"
OUT = PROJECT / "experiments/coffee17-w0-hbp-matched-v1"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 development folds.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_w0_hbp_matched_original.zip"
PROV_LOCAL = WORK / "coffee17_w0_hbp_matched_provenance"
CANONICAL = WORK / "coffee17_w0_hbp_matched_original_v1"
FOLDS_LOCAL = WORK / "coffee17_w0_hbp_matched_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV_LOCAL,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

# ------------------------------------------------------------
# 3. Five matched folds: retrain R0-HBP + train W0-HBP.
# ------------------------------------------------------------
print("\n=== 2/4 MATCHED R0-HBP VS W0-HBP ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_w0_hbp_matched_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_w0_hbp_matched",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"W0_HBP_MATCHED_fold{fold}.log")

    result_path = OUT / f"fold_{fold}" / "seed42" / "pair_result.json"
    result = json.loads(result_path.read_text(encoding="utf-8"))
    d = result["DELTA_W0_MINUS_R0"]

    print(
        f"FOLD {fold} | "
        f"R0-HBP Macro={result['R0_HBP']['macro_f1']:.4f} | "
        f"W0-HBP Macro={result['W0_HBP']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Paired summary.
# ------------------------------------------------------------
print("\n=== 3/4 PAIRED SUMMARY ===")
SUMMARY = PROJECT / "analysis" / "w0_hbp_matched_summary.json"

run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_w0_hbp_matched_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]
delta = agg["DELTA_W0_MINUS_R0"]

print("\n=== KEY RESULT ===")
print("R0-HBP Macro-F1 :", f"{agg['R0_HBP']['macro_f1']['mean']:.4%}")
print("W0-HBP Macro-F1 :", f"{agg['W0_HBP']['macro_f1']['mean']:.4%}")
print("Delta Macro-F1  :", f"{delta['macro_f1']['mean']:+.4%}")
print("R0-HBP Hard-F1  :", f"{agg['R0_HBP']['hard_class_f1']['mean']:.4%}")
print("W0-HBP Hard-F1  :", f"{agg['W0_HBP']['hard_class_f1']['mean']:.4%}")
print("Delta Hard-F1   :", f"{delta['hard_class_f1']['mean']:+.4%}")
print("R0-HBP Worst-F1 :", f"{agg['R0_HBP']['worst_class_f1']['mean']:.4%}")
print("W0-HBP Worst-F1 :", f"{agg['W0_HBP']['worst_class_f1']['mean']:.4%}")
print("Delta Worst-F1  :", f"{delta['worst_class_f1']['mean']:+.4%}")
print("Positive Macro folds:", summary["positive_delta_folds"]["macro_f1"], "/ 5")
print("Positive Hard folds :", summary["positive_delta_folds"]["hard_class_f1"], "/ 5")
print("Positive Worst folds:", summary["positive_delta_folds"]["worst_class_f1"], "/ 5")
print("SCREENING GATE      :", summary["screening_gate"]["decision"])

# ------------------------------------------------------------
# 5. Compact analysis package.
# ------------------------------------------------------------
print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")

PKG = WORK / "w0-hbp-matched-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")

for fold in range(1, 6):
    pair_src = OUT / f"fold_{fold}" / "seed42"
    pair_dst = PKG / f"fold_{fold}"
    pair_dst.mkdir(parents=True, exist_ok=True)

    shutil.copy2(pair_src / "pair_result.json", pair_dst / "pair_result.json")

    for arm in ("R0_HBP", "W0_HBP"):
        src = pair_src / arm
        dst = pair_dst / arm
        dst.mkdir(parents=True, exist_ok=True)

        for name in ("run_contract.json", "history.json", "run_config.yaml"):
            p = src / name
            if p.is_file():
                shutil.copy2(p, dst / name)

        if (src / "validation").is_dir():
            shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(CONFIG, PKG / "W0_HBP_MATCHED.yaml")
protocol = REPO / "docs/protocols/W0_HBP_MATCHED_V1.md"
if protocol.is_file():
    shutil.copy2(protocol, PKG / "W0_HBP_MATCHED_V1.md")

zip_path = Path(
    shutil.make_archive(
        str(WORK / "w0-hbp-matched-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("PROJECT:", PROJECT)
print("R0-HBP was retrained as a matched control.")
print("W0-HBP was trained as the candidate.")
print("Outer test was NOT accessed.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
